# Inflation Nowcasting — SOLUTION KEY
Reference implementations, one section per milestone. **Do not read ahead of your
current milestone** — see the rules in ROADMAP.md. Code favors clarity over generality.

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
import matplotlib.pyplot as plt

pd.set_option("display.precision", 3)
plt.rcParams["figure.figsize"] = (10, 3.5)

## M0 — Data (provided)

In [ ]:
# --- M0 (provided): data download -------------------------------------------
# FRED's plain CSV endpoint needs no API key. Runs on your machine with internet.

def fetch_fred(code: str, start: str = "1993-01-01") -> pd.Series:
    url = f"https://fred.stlouisfed.org/graph/fredgraph.csv?id={code}"
    df = pd.read_csv(url, index_col=0, parse_dates=True)
    s = pd.to_numeric(df.iloc[:, 0], errors="coerce").dropna()
    s.name = code
    return s[s.index >= start]

CODES = {
    "cpi":      "CPIAUCSL",        # CPI all items, SA, monthly
    "core_cpi": "CPILFESL",        # CPI ex food & energy, SA, monthly
    "food_cpi": "CPIUFDSL",        # CPI food, SA, monthly
    "gas_cpi":  "CUSR0000SETB01",  # CPI gasoline (all types), SA, monthly
    "pce":      "PCEPI",           # PCE price index, monthly
    "core_pce": "PCEPILFE",        # core PCE price index, monthly
    "gas_wk":   "GASALLW",         # EIA retail gasoline, all grades, NSA, weekly
    "oil":      "DCOILBRENTEU",    # Brent crude spot, daily
}

LVL = {k: fetch_fred(v) for k, v in CODES.items()}
for k, s in LVL.items():
    print(f"{k:9s} {s.index[0].date()} -> {s.index[-1].date()}  n={len(s)}")

In [ ]:
# ### Self-test M0 (provided — do not edit) ###
assert len(LVL) == 8
for k in ["cpi", "core_cpi", "food_cpi", "gas_cpi", "pce", "core_pce"]:
    assert LVL[k].index.freqstr or (LVL[k].index.to_series().diff().dt.days.dropna().between(28, 31).mean() > 0.95), k
assert LVL["gas_wk"].index.to_series().diff().dt.days.dropna().mode()[0] == 7
assert LVL["oil"].index.to_series().diff().dt.days.dropna().mode()[0] == 1
fig, axes = plt.subplots(2, 4, figsize=(14, 5))
for ax, (k, s) in zip(axes.flat, LVL.items()):
    s.plot(ax=ax, title=k)
plt.tight_layout(); plt.show()
print("M0 OK — 6 monthly, 1 weekly, 1 daily series")

### ✍️ Checkpoint M0 — answer in writing before continuing

Why does the paper use SA data everywhere except weekly retail gasoline? What breaks if you mix SA and NSA in the same regression?

**My answers:**

- 


## M1 — Inflation arithmetic (paper eqs 1–2)

In [ ]:
def monthly_inflation(levels: pd.Series) -> pd.Series:
    """Month-over-month inflation in PERCENT (not annualized)."""
    return 100 * levels.pct_change()


def quarterly_annualized(levels: pd.Series) -> pd.Series:
    """Quarterly annualized inflation per paper eqs (1)-(2)."""
    q = levels.groupby(levels.index.to_period("Q")).mean()
    return 100 * ((q / q.shift(1)) ** 4 - 1)

In [ ]:
# ### Self-test M1 (provided — do not edit) ###
# Cross-check against BEA's own quarterly PCE price index (independent series).
pcectpi = fetch_fred("PCECTPI")
bea_q = (100 * ((pcectpi / pcectpi.shift(1)) ** 4 - 1)).dropna()
bea_q.index = bea_q.index.to_period("Q")
mine = quarterly_annualized(LVL["pce"]).dropna()
common = mine.index.intersection(bea_q.index)[-20:]
gap = (mine[common] - bea_q[common]).abs()
print(gap.describe())
assert gap.max() < 0.2, "quarterly aggregation does not match BEA convention"
print("M1 OK — your eq(1)-(2) matches BEA's published quarterly PCE inflation")

### ✍️ Checkpoint M1 — answer in writing before continuing

1. Why does averaging three monthly *levels* differ from averaging three monthly *rates*?
2. Which months of quarter T−1 still influence quarter T inflation, and why does that give the model a head start before T begins?

**My answers:**

- 


## M2 — Core & food default: recursive MA12 (paper eq 4)

In [ ]:
def ma12_forecast(infl: pd.Series, n_ahead: int = 1) -> np.ndarray:
    """Paper eq (4): RECURSIVE 12-month moving average forecast."""
    hist = list(infl.dropna().iloc[-12:])
    out = []
    for _ in range(n_ahead):
        f = float(np.mean(hist[-12:]))
        out.append(f)
        hist.append(f)
    return np.array(out)

In [ ]:
# ### Self-test M2 (provided — do not edit) ###
const = pd.Series(0.2, index=pd.date_range("2000-01-01", periods=24, freq="MS"))
assert np.allclose(ma12_forecast(const, 6), 0.2), "constant series must forecast itself"

core = monthly_inflation(LVL["core_cpi"]).dropna()
eval_idx = core.loc["2005":"2019"].index
err_ma, err_rw = [], []
for t in eval_idx:
    hist = core.loc[:t].iloc[:-1]          # info: data through t-1 only
    err_ma.append(core[t] - ma12_forecast(hist, 1)[0])
    err_rw.append(core[t] - hist.iloc[-1])
rmse = lambda e: float(np.sqrt(np.mean(np.square(e))))
print(f"RMSE  MA12: {rmse(err_ma):.4f}   random walk: {rmse(err_rw):.4f}")
assert rmse(err_ma) < rmse(err_rw), "MA12 should beat the monthly random walk (Atkeson-Ohanian logic)"
print("M2 OK")

### ✍️ Checkpoint M2 — answer in writing before continuing

Why is this naive rule so hard to beat for core inflation at short horizons? What time-series property of core inflation does it exploit?

**My answers:**

- 


## M3 — The gasoline block (paper eqs 6–7 + footnote 7) — hardest milestone

In [ ]:
def weekly_to_monthly(weekly: pd.Series) -> pd.Series:
    """Average weekly observations within each calendar month."""
    m = weekly.groupby(weekly.index.to_period("M")).mean()
    m.index = m.index.to_timestamp()          # month-start dates, like FRED
    return m


def seasonally_adjust_gas(nsa_infl: pd.Series, cpi_gas_infl: pd.Series) -> pd.Series:
    """Paper footnote 7: sf = 3-yr average of (NSA - SA CPI gas) for that calendar month."""
    diff = (nsa_infl - cpi_gas_infl).dropna()
    sf = diff.groupby(diff.index.month).transform(lambda x: x.shift(1).rolling(3).mean())
    return (nsa_infl - sf).dropna()


def gas_forecast_from_oil(gas_m: pd.Series, oil_m: pd.Series, oil_hat: float,
                          window: int = 60) -> float:
    """Paper eqs (6)-(7): long-run relation + error-correction step."""
    df = pd.concat({"gas": gas_m, "oil": oil_m}, axis=1).dropna().iloc[-window:]
    lr = sm.OLS(df["gas"], sm.add_constant(df["oil"])).fit()          # eq (6)
    gap = df["gas"] - lr.fittedvalues
    d = pd.DataFrame({"dgas": df["gas"].diff(),
                      "doil": df["oil"].diff(),
                      "gap1": gap.shift(1)}).dropna()
    ecm = sm.OLS(d["dgas"], d[["doil", "gap1"]]).fit()                # eq (7)
    dgas_hat = (ecm.params["doil"] * (oil_hat - df["oil"].iloc[-1])
                + ecm.params["gap1"] * gap.iloc[-1])
    return float(df["gas"].iloc[-1] + dgas_hat)

In [ ]:
# ### Self-test M3 (provided — do not edit) ###
gas_m = weekly_to_monthly(LVL["gas_wk"])                    # $ / gallon, monthly avg
gas_nsa_infl = monthly_inflation(gas_m)
cpi_gas_infl = monthly_inflation(LVL["gas_cpi"])
gas_sa = seasonally_adjust_gas(gas_nsa_infl, cpi_gas_infl)

common = gas_sa.loc["2000":"2019"].index.intersection(cpi_gas_infl.index)
corr = gas_sa[common].corr(cpi_gas_infl[common])
print(f"corr(SA EIA gasoline inflation, CPI gasoline inflation) = {corr:.3f}")
assert corr > 0.9, "EIA-based SA gasoline inflation should track CPI gasoline closely"

oil_m = weekly_to_monthly(LVL["oil"])                       # daily -> monthly avg
oil_hat = float(LVL["oil"].iloc[-1])                        # random-walk oil forecast
lr_beta = sm.OLS(*[x.iloc[-60:] for x in
                   [pd.concat({"g": gas_m, "o": oil_m}, axis=1).dropna()["g"],
                    sm.add_constant(pd.concat({"g": gas_m, "o": oil_m}, axis=1).dropna()["o"])]]).fit().params.iloc[1]
assert lr_beta > 0, "long-run oil coefficient must be positive"
fcst = gas_forecast_from_oil(gas_m, oil_m, oil_hat)
print(f"next-month gasoline level forecast: {fcst:.2f} (last actual {gas_m.iloc[-1]:.2f})")
assert 0.5 * gas_m.iloc[-1] < fcst < 1.5 * gas_m.iloc[-1], "forecast should be near the last level"
print("M3 OK")

### ✍️ Checkpoint M3 — answer in writing before continuing

1. Why seasonally adjust with CPI-gasoline seasonal factors rather than running X-13 on the EIA series?
2. What pump-price behavior does the ECM's gap term capture?
3. Why does the paper use a random walk for oil instead of futures prices (see its footnote 9)?

**My answers:**

- 


## M4 — Bridge equations (paper eqs 5 & 9)

In [ ]:
def bridge_forecast(y_infl: pd.Series, x_infl: pd.Series, window: int = 24) -> float:
    """Paper eqs (5)/(9): same-month bridge, e.g. released CPI -> unreleased PCE."""
    df = pd.concat({"y": y_infl, "x": x_infl}, axis=1).dropna().iloc[-window:]
    fit = sm.OLS(df["y"], sm.add_constant(df["x"])).fit()
    x_new = float(x_infl.dropna().iloc[-1])       # month where x is out, y is not
    return float(fit.params.iloc[0] + fit.params.iloc[1] * x_new)

In [ ]:
# ### Self-test M4 (provided — do not edit) ###
core_cpi_i = monthly_inflation(LVL["core_cpi"]).dropna()
core_pce_i = monthly_inflation(LVL["core_pce"]).dropna()
eval_idx = core_pce_i.loc["2005":"2019"].index
err_br, err_ma = [], []
for t in eval_idx:
    y_hist = core_pce_i.loc[:t].iloc[:-1]         # PCE not yet out for month t
    x_hist = core_cpi_i.loc[:t]                   # CPI IS out for month t
    err_br.append(core_pce_i[t] - bridge_forecast(y_hist, x_hist))
    err_ma.append(core_pce_i[t] - ma12_forecast(y_hist, 1)[0])
rmse = lambda e: float(np.sqrt(np.mean(np.square(e))))
print(f"RMSE  bridge: {rmse(err_br):.4f}   MA12: {rmse(err_ma):.4f}")
assert rmse(err_br) < rmse(err_ma), "bridging released core CPI should beat MA12 for core PCE (paper Table 7, line 14)"
print("M4 OK")

### ✍️ Checkpoint M4 — answer in writing before continuing

1. What do CPI and PCE share that makes a same-month bridge work? Name two reasons they diverge.
2. Why a 24-month window instead of the full sample? (Connect to the bias-variance / regime-change argument.)

**My answers:**

- 


## M5 — Headline regression + deterministic model switching (paper eqs 8–11)

In [ ]:
def choose_regime(have_own_cpi_t: bool, have_gas_t: bool) -> str:
    """Deterministic model switching (eqs 9-11)."""
    if have_own_cpi_t:
        return "bridge"        # eq (9): CPI out, bridge to PCE
    if have_gas_t:
        return "disagg_reg"    # eq (10): gasoline nowcast available
    return "ma"                # eq (11): fall back to moving average


def headline_reg_forecast(y_infl: pd.Series, core_infl: pd.Series,
                          food_infl: pd.Series, gas_infl: pd.Series,
                          core_hat: float, food_hat: float, gas_hat: float,
                          window: int = 24) -> float:
    """Paper eq (10) for one headline measure."""
    df = pd.concat({"y": y_infl, "core": core_infl,
                    "food": food_infl, "gas": gas_infl}, axis=1).dropna().iloc[-window:]
    fit = sm.OLS(df["y"], sm.add_constant(df[["core", "food", "gas"]])).fit()
    b = fit.params
    return float(b["const"] + b["core"] * core_hat + b["food"] * food_hat + b["gas"] * gas_hat)

In [ ]:
# ### Self-test M5 (provided — do not edit) ###
assert choose_regime(True,  True)  == "bridge"
assert choose_regime(True,  False) == "bridge"
assert choose_regime(False, True)  == "disagg_reg"
assert choose_regime(False, False) == "ma"

cpi_i  = monthly_inflation(LVL["cpi"]).dropna()
food_i = monthly_inflation(LVL["food_cpi"]).dropna()
df = pd.concat({"y": cpi_i, "core": core_cpi_i, "food": food_i, "gas": gas_sa}, axis=1).dropna()
r2 = sm.OLS(df["y"], sm.add_constant(df[["core", "food", "gas"]])).fit().rsquared
print(f"full-sample R2 of eq(10) CPI regression: {r2:.3f}")
assert r2 > 0.5, "core+food+gas should explain most headline CPI variation"
print("M5 OK")

### ✍️ Checkpoint M5 — answer in writing before continuing

Write the regime table from memory: for information states (i) nothing released, no gas; (ii) weekly gas only; (iii) CPI(t) released — which equation governs, and why is this 'time-varying weights' in the eq (3) sense?

**My answers:**

- 


## M6 — Calendar simulation

We evaluate the model at three information dates ("cases") for each target month *t* — a simplified version of the paper's six (Table 2):

| Case | Stand-in date            | CPI thru | PCE thru | EIA gas in month t | Oil        |
|------|--------------------------|----------|----------|--------------------|------------|
| A    | last day of month t−1    | t−2      | t−2      | none               | thru t−1   |
| B    | ~15th of month t         | t−1      | t−2      | first 2 weeks      | thru day 14|
| C    | day after CPI(t) release | t        | t−1      | all weeks          | all        |

Your job: implement `nowcast_month(t, case)` returning nowcasts for all four measures,
respecting ONLY the information in the table (this discipline is the whole exercise).
Then the provided loop computes RMSEs per case. Deviation from the paper (noted in
ROADMAP): we evaluate 2005–2019 on final-vintage data, so error LEVELS won't match the
paper's tables — the declining-RMSE PATTERN is the success criterion.

In [ ]:
# Realized monthly inflation series (info truncation happens inside nowcast_month)
INFL = {
    "cpi":      monthly_inflation(LVL["cpi"]).dropna(),
    "core_cpi": monthly_inflation(LVL["core_cpi"]).dropna(),
    "food":     monthly_inflation(LVL["food_cpi"]).dropna(),
    "pce":      monthly_inflation(LVL["pce"]).dropna(),
    "core_pce": monthly_inflation(LVL["core_pce"]).dropna(),
    "gas_sa":   gas_sa,
}
GAS_M, OIL_D = gas_m, LVL["oil"]


def _ma_chain(hist: pd.Series, months_missing: int) -> float:
    """Recursive MA12 forecast for the last of `months_missing` missing months."""
    return ma12_forecast(hist, months_missing)[-1]


def nowcast_month(t: pd.Timestamp, case: str) -> dict:
    m = pd.DateOffset(months=1)
    cpi_thru = {"A": t - 2 * m, "B": t - m, "C": t}[case]
    pce_thru = {"A": t - 2 * m, "B": t - 2 * m, "C": t - m}[case]

    # --- step 1: truncated information sets ---------------------------------
    cpi_i   = INFL["cpi"].loc[:cpi_thru]
    ccpi_i  = INFL["core_cpi"].loc[:cpi_thru]
    food_i  = INFL["food"].loc[:cpi_thru]
    pce_i   = INFL["pce"].loc[:pce_thru]
    cpce_i  = INFL["core_pce"].loc[:pce_thru]

    # --- step 2: core CPI and food ------------------------------------------
    n_miss_cpi = (t.year - cpi_thru.year) * 12 + (t.month - cpi_thru.month)
    core_cpi_hat = ccpi_i[t] if n_miss_cpi == 0 else _ma_chain(ccpi_i, n_miss_cpi)
    food_hat     = food_i[t] if n_miss_cpi == 0 else _ma_chain(food_i, n_miss_cpi)

    # --- step 3: core PCE ----------------------------------------------------
    n_miss_pce = (t.year - pce_thru.year) * 12 + (t.month - pce_thru.month)
    if n_miss_pce == 0:
        core_pce_hat = cpce_i[t]
    elif cpi_thru > pce_thru:                       # CPI one month ahead: bridge, then chain
        bridged = bridge_forecast(cpce_i, ccpi_i)   # fills month cpi_thru
        if cpi_thru == t:
            core_pce_hat = bridged
        else:
            ext = pd.concat([cpce_i, pd.Series([bridged], index=[cpi_thru])])
            core_pce_hat = _ma_chain(ext, n_miss_cpi)
    else:
        core_pce_hat = _ma_chain(cpce_i, n_miss_pce)

    # --- step 4: gasoline -----------------------------------------------------
    sf = (monthly_inflation(GAS_M) - INFL["gas_sa"])          # realized seasonal factor series
    sf_t = sf[(sf.index.month == t.month) & (sf.index < t)].iloc[-3:].mean()
    weeks_t = LVL["gas_wk"].loc[(LVL["gas_wk"].index >= t) & (LVL["gas_wk"].index < t + m)]
    if case == "B":
        weeks_t = weeks_t[weeks_t.index.day <= 14]
    if case == "A" or len(weeks_t) == 0:
        gas_hist = GAS_M.loc[:t - m]
        oil_m_hist = weekly_to_monthly(OIL_D.loc[:t - m])
        oil_hat = float(OIL_D.loc[:t - m].iloc[-1])           # random-walk oil forecast
        gas_lvl_hat = gas_forecast_from_oil(gas_hist, oil_m_hist, oil_hat)
    else:
        gas_lvl_hat = float(weeks_t.mean())
    gas_nsa_hat = 100 * (gas_lvl_hat / GAS_M.loc[:t - m].iloc[-1] - 1)
    gas_hat = gas_nsa_hat - sf_t

    # --- step 5: headline ------------------------------------------------------
    gas_i_hist = INFL["gas_sa"].loc[:t - m]
    if case == "C":
        cpi_hat = cpi_i[t]                                     # released
        pce_hat = bridge_forecast(pce_i, cpi_i)                # eq (9)
    else:
        cpi_hat = headline_reg_forecast(cpi_i, ccpi_i, food_i, gas_i_hist,
                                        core_cpi_hat, food_hat, gas_hat)
        pce_hat = headline_reg_forecast(pce_i, cpce_i, food_i, gas_i_hist,
                                        core_pce_hat, food_hat, gas_hat)
    return {"cpi": cpi_hat, "core_cpi": core_cpi_hat,
            "pce": pce_hat, "core_pce": core_pce_hat}

In [ ]:
# ### Self-test M6 (provided — do not edit) ###
targets = INFL["pce"].loc["2005":"2019"].index
results = {}
for case in ["A", "B", "C"]:
    errs = {k: [] for k in ["cpi", "core_cpi", "pce", "core_pce"]}
    for t in targets:
        try:
            nc = nowcast_month(t, case)
        except Exception:
            continue
        for k in errs:
            errs[k].append(nc[k] - INFL[k][t])
    results[case] = {k: float(np.sqrt(np.mean(np.square(v)))) for k, v in errs.items()}
tbl = pd.DataFrame(results).T
print(tbl.round(4))

tbl[["cpi", "core_cpi"]].plot(kind="bar", title="Monthly nowcast RMSE by case (paper Fig. 2 analogue)")
plt.ylabel("pp, month-over-month"); plt.show()

assert tbl.loc["C", "cpi"] < tbl.loc["B", "cpi"] < tbl.loc["A", "cpi"], \
    "headline CPI RMSE must fall as information arrives"
assert (tbl.loc["A", "core_cpi"] - tbl.loc["B", "core_cpi"]) < \
       (tbl.loc["A", "cpi"] - tbl.loc["B", "cpi"]), \
    "core should improve far less than headline (gasoline is doing the work)"
print("M6 OK — you have replicated the paper's signature result")

### ✍️ Checkpoint M6 — answer in writing before continuing

Explain in one sentence why core RMSE barely moves from case A to C while headline RMSE falls sharply.

**My answers:**

- 


## M7 — Stretch goals (see ROADMAP.md)

Pick any: ALFRED real-time vintages · Diebold–Mariano tests · ML horse-race
(ridge / gradient boosting vs eq-10 OLS on the same harness) · quarterly aggregation
(reuse `quarterly_annualized` and the paper's 14 cases).

Before extending, answer in writing: *which of the paper's design choices would you
keep in an ML version, and why?* (Expected: the calendar discipline, the real-time
evaluation, the short windows — the model class is the least important part.)